# TP 3 — Matplotlib & histogrammes

**Objectifs**

- Maîtriser l'affichage d'images avec Matplotlib (figures, sous-graphes, titres).
- Calculer et tracer l'histogramme d'intensité d'une image en niveaux de gris et l'histogramme par canal d'une image RGB.
- Implémenter manuellement deux techniques d'amélioration de contraste :
  - étirement linéaire par percentiles ;
  - égalisation d'histogramme via la fonction de répartition cumulative.
- Comparer visuellement et statistiquement l'effet de ces traitements.

**Durée indicative : 50 minutes.**

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from skimage import data

# 'moon' est une image en niveaux de gris peu contrastée — idéale pour ce TP.
img_gray = np.array(Image.fromarray(data.moon()))  # (H, W), uint8
img_rgb = np.array(Image.fromarray(data.coffee()))  # (H, W, 3), uint8

print("gray :", img_gray.shape, img_gray.dtype, img_gray.min(), img_gray.max())
print("rgb  :", img_rgb.shape, img_rgb.dtype, img_rgb.min(), img_rgb.max())

gray : (512, 512) uint8 0 255
rgb  : (400, 600, 3) uint8 0 255


## Exercice 1 — Affichage propre et colorbar

Affichez `img_gray` et `img_rgb` côte à côte avec :

- une figure de taille raisonnable (`figsize=(10, 4)`),
- un titre pour chaque sous-graphe,
- les axes désactivés,
- `cmap="gray"` et `vmin=0, vmax=255` pour l'image en niveaux de gris.

**Ajout — colorbar avec une colormap perceptuelle** : créez ensuite **une troisième figure** affichant `img_gray` avec la colormap `"viridis"` et **une colorbar visible à droite** (`plt.colorbar(im, ax=ax)`). C'est l'affichage type quand on visualise une grandeur scalaire (carte de profondeur, heatmap d'activation, intensité d'un capteur).

<details>
<summary>💡 Coup de pouce — affichage propre et colorbar</summary>

**🎯 But :** une figure « propre » qui se lit instantanément. Sans colorbar, on ne sait pas à quoi correspondent les couleurs.

**Récupérer l'objet `imshow` pour la colorbar**

`imshow` retourne un objet matplotlib (un `AxesImage`) qui contient les infos de mapping valeur↔couleur. Sans cet objet, `plt.colorbar` ne sait pas quelles couleurs afficher. Donc on le capture :

```python
im = ax.imshow(img, cmap="viridis")   # garde la référence !
plt.colorbar(im, ax=ax)                # OK, im contient la colormap
```

Si on écrit juste `ax.imshow(img)` sans capturer, `plt.colorbar()` plantera ou affichera n'importe quoi.

**Ajuster la taille de la colorbar**

Par défaut elle est énorme et mal alignée :

```python
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
```

- `fraction=0.046` → barre fine (4.6% de la largeur de l'axe)
- `pad=0.04` → petit espace entre image et barre

C'est une formule magique standard qu'on retrouve partout dans le code matplotlib pédagogique.

**Conventions à appliquer toujours**
- `ax.set_title("...")` pour le titre
- `ax.axis("off")` si c'est une image (pas une vraie courbe)
- `fig.tight_layout()` pour éviter les chevauchements

</details>

In [2]:
# TODO : affichage côte à côte img_gray + img_rgb

# TODO : 2e figure avec colormap "viridis" + colorbar
# fig, ax = plt.subplots(figsize=(5, 4))
# im = ax.imshow(img_gray, cmap="viridis")
# plt.colorbar(im, ax=ax)

## Exercice 2 — Histogramme en niveaux de gris

1. Calculez l'histogramme de `img_gray` avec `np.histogram` (256 bins, plage `[0, 256]`).
2. Tracez-le en bâtonnets (`plt.bar`) ou en ligne continue (`plt.plot`).
3. Constatez visuellement que l'histogramme est concentré dans une plage étroite : `img_gray` est peu contrastée.

<details>
<summary>💡 Coup de pouce — histogramme en niveaux de gris</summary>

**🎯 But :** compter combien de pixels ont chaque intensité possible. C'est la base de toutes les analyses de contraste.

**Calcul avec NumPy**

```python
hist, edges = np.histogram(img, bins=256, range=(0, 256))
```

- `hist` : tableau de **256 entiers** (combien de pixels pour chaque valeur 0, 1, ..., 255)
- `edges` : tableau de **257 valeurs** (les bornes des bins : `[0, 1, 2, ..., 256]`)
- Attention : `edges` a **une valeur de plus** que `hist` (les bornes encadrent les bins).

⚠️ **Toujours préciser `range=(0, 256)`**. Sans ça, NumPy ajuste les bornes aux valeurs présentes dans l'image → si vous comparez deux images, elles auront des bins différents et les graphes ne seront pas comparables.

**Tracer avec `plt.bar`**

Il faut donner les **centres** des bins (pas les bornes) :

```python
centers = (edges[:-1] + edges[1:]) / 2
ax.bar(centers, hist, width=1)
```

**Alternative plus simple — `plt.plot`**

Pour des intensités entières 0-255, l'index = la valeur :

```python
ax.plot(hist)   # x implicite : 0, 1, 2, ..., 255 = les intensités
```

Tracer en courbe perd un peu en fidélité (l'histogramme est discret) mais c'est très lisible et rapide à coder.

**Interpréter** : une bosse sombre = beaucoup de pixels noirs (sous-exposé). Deux bosses séparées = histogramme **bimodal**, idéal pour seuillage.

</details>

In [3]:
# TODO

## Exercice 3 — Histogramme par canal RGB

Sur une même figure, tracez les histogrammes des canaux rouge, vert et bleu de `img_rgb`, en utilisant la couleur correspondante pour chaque courbe.

<details>
<summary>💡 Coup de pouce — 3 histogrammes RGB superposés</summary>

**🎯 But :** voir d'un coup d'œil la distribution de chaque canal. Une image trop rouge se verra immédiatement : la courbe rouge sera décalée vers la droite.

**Boucle élégante**

```python
for i, color in enumerate(["red", "green", "blue"]):
    channel = img_rgb[..., i]
    hist, _ = np.histogram(channel, bins=256, range=(0, 256))
    ax.plot(hist, color=color, label=color, alpha=0.7)
ax.legend()
```

- `enumerate(["red", "green", "blue"])` parcourt `(0, "red"), (1, "green"), (2, "blue")` — combiner l'index et la couleur en une seule boucle.
- `color=color` dans `plt.plot` : Matplotlib comprend les noms de couleurs HTML standards.
- `alpha=0.7` : transparence pour bien voir les chevauchements quand les courbes se croisent.
- `label=color` + `ax.legend()` : légende automatique.

**Conseil de présentation**

Si les valeurs des histogrammes sont très différentes (par exemple une image quasi-monochrome rouge), pensez à `ax.set_yscale("log")` pour voir aussi les petites contributions.

**Astuce — version `seaborn.histplot`**

Pour les utilisateurs de Seaborn : `sns.histplot` accepte directement les données 1D et gère légende, colors et transparence en une ligne — mais nécessite d'importer Seaborn.

</details>

In [4]:
# TODO

## Exercice 4 — Étirement de contraste par percentiles

Implémentez la fonction suivante :

```python
def stretch(img, p_low=2, p_high=98):
    """Étire le contraste d'une image uint8 sur la plage des percentiles
    [p_low, p_high] vers [0, 255], avec clipping."""
```

Appliquez-la à `img_gray` et tracez :

- l'image d'origine et l'image étirée côte à côte,
- leurs histogrammes respectifs en-dessous (figure 2×2).

<details>
<summary>💡 Coup de pouce — étirement de contraste par percentiles</summary>

**🎯 But :** étirer l'histogramme pour qu'il occupe toute la palette [0, 255]. Plus robuste qu'un simple `min`/`max` (qui se fait piéger par 1 seul pixel aberrant).

**Calcul des bornes**

```python
a, b = np.percentile(img, [2, 98])   # ignore les 2% les plus sombres et les 2% les plus clairs
```

- Les 2 % de pixels les plus sombres deviennent du **noir pur** (clip à 0).
- Les 2 % les plus clairs deviennent du **blanc pur** (clip à 255).
- Le reste s'étale linéairement entre 0 et 255.

C'est exactement ce que fait le bouton « auto-niveau » de Photoshop ou GIMP.

**Formule de stretching**

```python
out = (img - a) * 255 / (b - a)
```

Passez **d'abord en float** sinon le `img - a` peut déborder en uint8 négatif :

```python
out = (img.astype(np.float32) - a) * 255 / (b - a)
out = np.clip(out, 0, 255).astype(np.uint8)
```

- `np.clip` ramène les valeurs hors [0, 255] aux bornes (sans cette étape, le cast uint8 ferait des trucs bizarres par enroulement).
- `.astype(np.uint8)` à la fin pour récupérer un type image standard.

**Pourquoi les percentiles plutôt que min/max ?**

`min` et `max` sont **fragiles** : un seul pixel mort (capteur défectueux) à 0 ou un saturé à 255 et toute votre image se retrouve avec une plage normale → effet d'étirement nul. Les percentiles ignorent ces outliers et donnent un résultat robuste.

</details>

In [5]:
# TODO

## Exercice 5 — Égalisation d'histogramme manuelle

Implémentez l'égalisation d'histogramme via la fonction de répartition cumulative :

```python
def equalize(img):
    """Égalise l'histogramme d'une image uint8 en niveaux de gris."""
```

Étapes :

1. Calculer l'histogramme `hist` sur 256 bins.
2. Calculer sa CDF normalisée à `[0, 255]`.
3. Remapper l'image par indexation : `out = cdf[img]`.
4. Cast en `uint8`.

Comparez les trois résultats — originale, étirée, égalisée — sur une figure 2×3 (images en haut, histogrammes en bas).

**Bonus** : ajoutez une quatrième version « correction gamma » avec γ = 0.6 :  
`gamma(img, γ) = clip( 255 * (img / 255) ** γ , 0, 255 )`. Lequel des trois traitements préférez-vous, et pourquoi ?

<details>
<summary>💡 Coup de pouce — égalisation d'histogramme manuelle</summary>

**🎯 But :** redistribuer les intensités pour que **toutes les valeurs soient à peu près également fréquentes**. Effet : une image plate (histogramme groupé en bosse) devient contrastée (histogramme étalé).

**Principe en 3 étapes**

1. **Histogramme** : combien de pixels pour chaque intensité ?
2. **CDF** (Cumulative Distribution Function) : combien de pixels ont une valeur **inférieure ou égale** à `v` ?
3. **LUT** (Look-Up Table) : pour chaque ancienne valeur `v`, sa nouvelle valeur = `255 × CDF(v) / CDF(255)`.

**Code**

```python
hist, _ = np.histogram(img, bins=256, range=(0, 256))
cdf = hist.cumsum()                    # 1) cumul
cdf = (255 * cdf / cdf[-1]).astype(np.uint8)   # 2) normalise à [0, 255]
egal = cdf[img]                        # 3) magie : LUT via fancy indexing !
```

**L'astuce magique : `cdf[img]`**

`cdf` est un tableau 1D de 256 valeurs. `img` est un tableau 2D d'intensités 0-255. **NumPy permet d'indexer un tableau par un autre tableau** :
- `cdf[img]` produit un tableau de même shape que `img`
- Chaque pixel `img[i, j]` est remplacé par `cdf[img[i, j]]`

C'est l'équivalent d'une boucle pixel par pixel, mais vectorisé en C → **ultra rapide**.

**Bonus — correction gamma**

```python
g = 0.5   # < 1 éclaircit, > 1 assombrit
out = 255 * (img / 255) ** g
out = np.clip(out, 0, 255).astype(np.uint8)
```

- `img / 255` : normalise en [0, 1].
- `** g` : élévation à la puissance non-linéaire.
- `g = 1` : pas d'effet. `g < 1` : éclaircit les zones sombres (utile pour photo sous-exposée). `g > 1` : assombrit.

**Différence avec l'égalisation :**
- Gamma = **non-linéaire fixe** (formule constante quelle que soit l'image)
- Égalisation = **adaptative** (dépend de l'histogramme de l'image)

L'égalisation est plus efficace pour les images très déséquilibrées, gamma est plus prévisible et contrôlable.

</details>

In [6]:
# TODO